# 🚀 AntiGravBot — Chatbot Materi *Google Antigravity 2.0*
**Tugas Mata Kuliah — Pembangunan Chatbot AI**
**RENXY LICHANDY**  
Mahasiswa | 261012000110

Aplikasi web: https://antigravbot.streamlit.app

Notebook ini berisi seluruh mesin chatbot: basis pengetahuan dari infografis, retriever TF-IDF,
mode Gemini RAG (opsional), evaluasi akurasi, dan koneksi ke database PostgreSQL cloud (Supabase/Neon)
yang juga dipakai aplikasi Streamlit.

**Cara pakai:** klik 🔑 *Secrets* di panel kiri Colab → tambahkan `DATABASE_URL` (dan opsional
`GEMINI_API_KEY`) → aktifkan *Notebook access* → **Runtime ▸ Run all**.

## 1. Instalasi library

In [ ]:
!pip install -q google-genai sqlalchemy psycopg2-binary scikit-learn pandas matplotlib

## 2. Basis pengetahuan (dari infografis)
17 dokumen materi + 20 pertanyaan uji.

In [ ]:
"""
knowledge_base.py
Basis pengetahuan chatbot, disusun dari materi infografis
"Google Antigravity 2.0: Era Baru Rekayasa Perangkat Lunak Berbasis Agen AI".

Setiap dokumen punya:
  kode       : ID unik (dipakai sebagai primary key di database)
  topik      : kelompok materi
  judul      : judul singkat
  konten     : isi jawaban
  kata_kunci : sinonim/kata kunci tambahan agar pencarian lebih akurat
  sumber     : "Infografis" (langsung dari materi) atau
               "Penjelasan tambahan" (uraian pendukung dari isi infografis)
"""

INFO_MATERI = {
    "judul": "Google Antigravity 2.0: Era Baru Rekayasa Perangkat Lunak Berbasis Agen AI",
    "subjudul": (
        "Paradigma baru: platform semi-otonom untuk eksekusi perangkat lunak "
        "end-to-end dari perancangan hingga pengujian."
    ),
}

DOKUMEN = [
    {
        "kode": "AG-00",
        "topik": "Gambaran Umum",
        "judul": "Apa itu Google Antigravity 2.0",
        "kata_kunci": "google antigravity 2 0 antigravity antigravity pengertian definisi apa itu antigravity platform agen ai paradigma baru "
                      "semi otonom end to end rekayasa perangkat lunak software engineering",
        "konten": (
            "Google Antigravity 2.0 adalah platform rekayasa perangkat lunak berbasis agen AI "
            "yang disebut sebagai era baru pengembangan software. Paradigmanya adalah platform "
            "semi-otonom untuk mengeksekusi pengembangan perangkat lunak secara end-to-end, "
            "mulai dari perancangan hingga pengujian. Pengembang tidak lagi hanya menulis kode "
            "baris demi baris, tetapi mendelegasikan pekerjaan kepada agen AI, memantau kerjanya, "
            "lalu memverifikasi hasilnya."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-01",
        "topik": "4 Pilar Utama",
        "judul": "4 Pilar Utama Platform Antigravity 2.0",
        "kata_kunci": "empat pilar utama komponen fitur utama bagian platform struktur arsitektur",
        "konten": (
            "Infografis menyebut 4 pilar utama platform Antigravity 2.0:\n"
            "1. **Desktop Agent Manager** – aplikasi mandiri untuk mengorkestrasi dan memantau "
            "beberapa agen AI secara paralel di berbagai proyek.\n"
            "2. **Antigravity IDE & Ext** – lingkungan pengembangan berbasis AI serta ekstensi "
            "integrasi untuk VS Code, JetBrains, dan Zed.\n"
            "3. **CLI dan SDK Fleksibel** – antarmuka baris perintah dan kit pengembang untuk "
            "otomatisasi skrip kustom di sisi server.\n"
            "4. **Konfigurasi Agents & Skills** – definisi peran tim melalui berkas agents.md dan "
            "panduan instruksi modular dalam skills.md.\n\n"
            "Selain itu, hasil kerja agen diperiksa lewat **Verifikasi via Artifacts**."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-02",
        "topik": "4 Pilar Utama",
        "judul": "Desktop Agent Manager",
        "kata_kunci": "desktop agent manager aplikasi mandiri orkestrasi orkestrator memantau "
                      "monitoring banyak agen paralel multi proyek",
        "konten": (
            "Desktop Agent Manager adalah aplikasi mandiri (standalone) untuk mengorkestrasi dan "
            "memantau beberapa agen AI secara paralel di berbagai proyek. Dengan pilar ini, satu "
            "pengembang dapat menjalankan beberapa agen sekaligus, misalnya satu agen membuat "
            "fitur, satu menulis pengujian, dan satu memperbaiki bug, lalu memantau semuanya dari "
            "satu tempat."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-03",
        "topik": "4 Pilar Utama",
        "judul": "Antigravity IDE & Extension",
        "kata_kunci": "ide extension ekstensi plugin vs code vscode jetbrains intellij zed editor "
                      "lingkungan pengembangan integrasi",
        "konten": (
            "Antigravity IDE & Ext adalah lingkungan pengembangan (IDE) berbasis AI, ditambah "
            "ekstensi integrasi untuk editor populer: **VS Code, JetBrains, dan Zed**. Jadi "
            "pengembang bisa memakai kemampuan agen Antigravity tanpa harus meninggalkan editor "
            "yang sudah biasa dipakai."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-04",
        "topik": "4 Pilar Utama",
        "judul": "CLI dan SDK Fleksibel",
        "kata_kunci": "cli command line terminal baris perintah sdk software development kit "
                      "otomatisasi skrip kustom server side backend pipeline ci cd",
        "konten": (
            "CLI dan SDK Fleksibel terdiri dari antarmuka baris perintah (CLI) dan kit pengembang "
            "(SDK) untuk otomatisasi skrip kustom di sisi server (server-side). Pilar ini membuat "
            "agen bisa dipanggil dari terminal atau disisipkan ke dalam skrip dan alur otomatisasi "
            "sendiri, tidak hanya lewat tampilan grafis."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-05",
        "topik": "4 Pilar Utama",
        "judul": "Konfigurasi Agents & Skills",
        "kata_kunci": "konfigurasi agents md skills md agents.md skills.md peran tim role "
                      "instruksi modular berkas file markdown pengaturan agen",
        "konten": (
            "Konfigurasi Agents & Skills adalah cara mendefinisikan perilaku agen:\n"
            "- **agents.md** – berkas untuk mendefinisikan peran tim (misalnya agen frontend, "
            "agen backend, agen penguji).\n"
            "- **skills.md** – panduan instruksi yang bersifat modular, sehingga satu keahlian "
            "dapat dipakai ulang oleh agen yang berbeda.\n\n"
            "Karena berbentuk berkas teks, konfigurasi ini mudah dibagikan dan disimpan bersama "
            "kode proyek."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-06",
        "topik": "Verifikasi",
        "judul": "Verifikasi via Artifacts",
        "kata_kunci": "verifikasi artifacts artifacts artifacts artefak bukti hasil kerja transparan dokumen rencana "
                      "tangkapan layar screenshot rekaman pengujian visual video review",
        "konten": (
            "Verifikasi via Artifacts berarti hasil kerja agen disajikan secara transparan melalui "
            "**dokumen rencana, tangkapan layar, dan rekaman pengujian visual**. Pengembang tidak "
            "perlu membaca setiap baris kode yang ditulis agen; cukup memeriksa artefak tersebut "
            "untuk menilai apakah rencana dan hasilnya sudah benar sebelum diterima."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-07",
        "topik": "Mode Tampilan",
        "judul": "Editor View",
        "kata_kunci": "editor view tampilan editor ide tradisional vs code manual menulis kode "
                      "baris kode perbaikan langsung",
        "konten": (
            "Editor View adalah mode tampilan untuk pengembang yang bekerja langsung di kode:\n"
            "- **Fokus utama:** penulisan dan pengeditan kode secara manual.\n"
            "- **Tampilan antarmuka:** IDE tradisional (berbasis VS Code).\n"
            "- **Interaksi agen AI:** bantuan per baris kode dan perbaikan langsung."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-08",
        "topik": "Mode Tampilan",
        "judul": "Manager View",
        "kata_kunci": "manager view tampilan manajer dashboard dasbor pusat kontrol pengawasan "
                      "koordinasi makro delegasi tugas besar otomatisasi paralel",
        "konten": (
            "Manager View adalah mode tampilan untuk mengelola agen:\n"
            "- **Fokus utama:** pengawasan dan koordinasi agen AI secara makro.\n"
            "- **Tampilan antarmuka:** dasbor pusat kontrol mandiri.\n"
            "- **Interaksi agen AI:** pendelegasian tugas besar dan otomatisasi paralel."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-09",
        "topik": "Mode Tampilan",
        "judul": "Perbandingan Editor View vs Manager View",
        "kata_kunci": "perbandingan beda perbedaan bandingkan editor view manager view versus vs "
                      "mode tampilan interaksi pengembang",
        "konten": (
            "Perbandingan mode tampilan interaksi pengembang:\n\n"
            "| Aspek | Editor View | Manager View |\n"
            "|---|---|---|\n"
            "| Fokus utama | Penulisan & pengeditan kode manual | Pengawasan & koordinasi agen AI secara makro |\n"
            "| Tampilan antarmuka | IDE tradisional (berbasis VS Code) | Dasbor pusat kontrol mandiri |\n"
            "| Interaksi agen AI | Bantuan baris kode & perbaikan langsung | Delegasi tugas besar & otomatisasi paralel |\n\n"
            "Singkatnya, Editor View cocok saat pengembang ingin memegang kode sendiri, sedangkan "
            "Manager View cocok saat pengembang berperan sebagai manajer yang membagi tugas ke "
            "banyak agen."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-10",
        "topik": "Kebijakan Otonomi",
        "judul": "3 Tingkat Kebijakan Otonomi",
        "kata_kunci": "tiga tingkat kebijakan otonomi level mode off auto turbo izin kontrol "
                      "keamanan eksekusi moda",
        "konten": (
            "Antigravity 2.0 punya 3 tingkat kebijakan otonomi yang mengatur seberapa bebas agen "
            "bertindak:\n"
            "1. **Mode Off** – agen hanya memberi saran; eksekusi dilakukan manual oleh pengembang.\n"
            "2. **Mode Auto** – agen menjalankan eksekusi yang dinilai aman secara otomatis.\n"
            "3. **Mode Turbo** – otomatisasi eksekusi penuh oleh agen."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-11",
        "topik": "Kebijakan Otonomi",
        "judul": "Mode Off",
        "kata_kunci": "mode off moda off mati saran manual tanpa eksekusi otomatis kontrol penuh",
        "konten": (
            "Mode Off adalah tingkat otonomi terendah: agen hanya memberikan **saran manual**. "
            "Semua tindakan tetap dieksekusi sendiri oleh pengembang. Mode ini cocok untuk kode "
            "yang sensitif atau saat pengembang ingin kontrol penuh."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-12",
        "topik": "Kebijakan Otonomi",
        "judul": "Mode Auto",
        "kata_kunci": "mode auto moda auto otomatis eksekusi aman tengah seimbang",
        "konten": (
            "Mode Auto adalah tingkat otonomi menengah: agen boleh menjalankan **eksekusi yang aman** "
            "secara otomatis. Tindakan yang berisiko tetap memerlukan persetujuan pengembang. Mode "
            "ini menyeimbangkan kecepatan dan keamanan."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-13",
        "topik": "Kebijakan Otonomi",
        "judul": "Mode Turbo",
        "kata_kunci": "mode turbo moda turbo otomatisasi penuh eksekusi penuh cepat otonom penuh",
        "konten": (
            "Mode Turbo adalah tingkat otonomi tertinggi: **otomatisasi eksekusi penuh**. Agen "
            "menjalankan langkah-langkah tanpa menunggu persetujuan satu per satu. Mode ini paling "
            "cepat, tetapi sebaiknya dipakai di lingkungan yang aman (misalnya proyek uji coba) dan "
            "hasilnya tetap diperiksa lewat Artifacts."
        ),
        "sumber": "Infografis",
    },
    {
        "kode": "AG-14",
        "topik": "Penjelasan Tambahan",
        "judul": "Alur Kerja End-to-End dengan Antigravity",
        "kata_kunci": "alur kerja workflow langkah tahapan proses cara kerja bagaimana bekerja "
                      "perancangan pengujian end to end contoh penggunaan",
        "konten": (
            "Berdasarkan komponen di infografis, alur kerja end-to-end dapat digambarkan sebagai "
            "berikut:\n"
            "1. **Atur tim agen** lewat agents.md dan skills.md.\n"
            "2. **Pilih tingkat otonomi** (Off, Auto, atau Turbo).\n"
            "3. **Delegasikan tugas** dari Manager View / Desktop Agent Manager, atau kerjakan detail "
            "kode di Editor View / IDE.\n"
            "4. **Agen mengeksekusi** perancangan, penulisan kode, hingga pengujian (bisa paralel).\n"
            "5. **Verifikasi hasil** melalui Artifacts: dokumen rencana, tangkapan layar, dan rekaman "
            "pengujian.\n"
            "6. Untuk otomatisasi rutin di server, gunakan **CLI dan SDK**."
        ),
        "sumber": "Penjelasan tambahan",
    },
    {
        "kode": "AG-15",
        "topik": "Penjelasan Tambahan",
        "judul": "Perbedaan dengan Asisten Kode Tradisional & Manfaatnya",
        "kata_kunci": "manfaat keunggulan kelebihan perbedaan asisten kode autocomplete copilot "
                      "tradisional mengapa penting dampak produktivitas peran developer",
        "konten": (
            "Asisten kode tradisional umumnya membantu per baris (melengkapi atau memperbaiki kode). "
            "Antigravity 2.0 menggeser peran pengembang menjadi **pengarah dan pengawas agen**: "
            "tugas besar didelegasikan, beberapa agen bekerja paralel, dan hasilnya dibuktikan lewat "
            "Artifacts. Manfaat yang ditawarkan antara lain produktivitas lebih tinggi, pekerjaan "
            "paralel lintas proyek, fleksibilitas (IDE, ekstensi, CLI/SDK), serta kendali risiko "
            "melalui tingkat otonomi Off/Auto/Turbo."
        ),
        "sumber": "Penjelasan tambahan",
    },
    {
        "kode": "AG-16",
        "topik": "Penjelasan Tambahan",
        "judul": "Tantangan dan Tips Penggunaan yang Aman",
        "kata_kunci": "tantangan risiko kekurangan kelemahan keamanan tips aman saran best practice "
                      "praktik terbaik hati hati",
        "konten": (
            "Hal yang perlu diperhatikan saat memakai platform agen semi-otonom:\n"
            "- Semakin tinggi otonomi (Turbo), semakin penting pengawasan; mulai dari Mode Off/Auto.\n"
            "- Selalu periksa Artifacts (rencana, tangkapan layar, rekaman uji) sebelum menerima hasil.\n"
            "- Tulis agents.md dan skills.md dengan jelas agar agen tidak salah memahami peran.\n"
            "- Pengembang tetap bertanggung jawab atas kualitas dan keamanan kode akhir."
        ),
        "sumber": "Penjelasan tambahan",
    },
]

CONTOH_PERTANYAAN = [
    "Apa itu Google Antigravity 2.0?",
    "Sebutkan 4 pilar utama Antigravity",
    "Apa beda Editor View dan Manager View?",
    "Jelaskan 3 tingkat kebijakan otonomi",
    "Fungsi agents.md dan skills.md?",
    "Bagaimana hasil kerja agen diverifikasi?",
]

# Data uji untuk evaluasi akurasi retrieval (pertanyaan -> kode dokumen yang benar)
DATA_UJI = [
    ("apa sih antigravity itu", "AG-00"),
    ("pilar utama platform ada apa saja", "AG-01"),
    ("fungsi desktop agent manager", "AG-02"),
    ("bisa dipakai di vscode atau jetbrains?", "AG-03"),
    ("editor apa saja yang didukung ekstensinya", "AG-03"),
    ("apa kegunaan CLI dan SDK", "AG-04"),
    ("untuk apa file agents.md", "AG-05"),
    ("apa isi skills.md", "AG-05"),
    ("bagaimana cara memverifikasi hasil kerja agen", "AG-06"),
    ("apa itu artefak dalam antigravity", "AG-06"),
    ("jelaskan editor view", "AG-07"),
    ("apa itu manager view", "AG-08"),
    ("perbedaan editor view dengan manager view", "AG-09"),
    ("tingkat otonomi ada berapa", "AG-10"),
    ("mode turbo itu apa", "AG-13"),
    ("kapan pakai moda off", "AG-11"),
    ("mode auto", "AG-12"),
    ("bagaimana alur kerja dari perancangan sampai pengujian", "AG-14"),
    ("apa kelebihannya dibanding copilot", "AG-15"),
    ("apa risikonya, ada tips aman?", "AG-16"),
]

## 3. Mesin chatbot
- **Pra-pemrosesan**: huruf kecil, hapus tanda baca & stopword, samakan sinonim.
- **Retriever**: cosine similarity gabungan TF-IDF kata (0,6) + n-gram karakter (0,4).
- **Ambang 0,12**: di bawahnya dianggap di luar materi.
- **Gemini RAG**: dokumen teratas dikirim sebagai konteks ke Gemini (jika API key ada).

In [ ]:
"""
engine.py
Mesin chatbot AntiGravBot.

Dua mode jawaban:
  1. RETRIEVAL  : mencari dokumen paling relevan dengan TF-IDF (kata + karakter)
                  lalu menampilkan isinya. Tidak butuh API key, selalu jalan.
  2. GEMINI-RAG : dokumen hasil retrieval dikirim sebagai konteks ke Google Gemini
                  (Retrieval-Augmented Generation) agar jawaban lebih natural.
                  Jika API key kosong / error, otomatis kembali ke mode RETRIEVAL.
"""

import re

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# Pra-pemrosesan teks Bahasa Indonesia
# ---------------------------------------------------------------------------
STOPWORDS = {
    "yang", "dan", "di", "ke", "dari", "apa", "itu", "ini", "adalah", "untuk", "dengan",
    "pada", "dalam", "saya", "aku", "kamu", "anda", "apakah", "bagaimana", "gimana",
    "jelaskan", "tolong", "sebutkan", "mengenai", "tentang", "sih", "dong", "ya", "kah",
    "nya", "atau", "juga", "bisa", "ada", "saja", "the", "a", "of", "is", "what",
    "mohon", "coba", "berikan", "boleh", "kan", "lah", "pun", "akan", "sebagai",
}

SINONIM = {
    "agent": "agen", "agents": "agen", "artefak": "artifacts", "artifact": "artifacts",
    "moda": "mode", "dasbor": "dashboard", "dasboard": "dashboard", "vscode": "vs code",
    "jet": "jetbrains", "intellij": "jetbrains", "screenshot": "tangkapan layar",
    "beda": "perbedaan", "bedanya": "perbedaan", "membedakan": "perbedaan",
    "kelebihan": "manfaat", "kelebihannya": "manfaat", "keunggulan": "manfaat",
    "risiko": "risiko tantangan", "risikonya": "risiko tantangan",
    "workflow": "alur kerja", "otomatis": "auto", "terminal": "cli",
}


def normalisasi(teks: str) -> str:
    teks = teks.lower()
    teks = teks.replace("agents.md", "agents md").replace("skills.md", "skills md")
    teks = re.sub(r"[^a-z0-9\s]", " ", teks)
    token = []
    for t in teks.split():
        t = SINONIM.get(t, t)
        token.extend(x for x in t.split() if x not in STOPWORDS)
    return " ".join(token)


# ---------------------------------------------------------------------------
# Jawaban untuk sapaan / percakapan ringan (intent sederhana)
# ---------------------------------------------------------------------------
INTENT = [
    (r"^(hai|halo|hallo|hi|hello|hey|selamat (pagi|siang|sore|malam)|assalamualaikum)\b",
     "Halo! 👋 Saya **AntiGravBot**, asisten belajar materi *Google Antigravity 2.0*. "
     "Silakan tanya tentang 4 pilar platform, Editor View vs Manager View, "
     "Artifacts, atau tingkat otonomi Off/Auto/Turbo."),
    (r"(terima ?kasih|makasih|thanks|thank you|thx)",
     "Sama-sama! Senang bisa membantu. Ada lagi yang ingin ditanyakan tentang Antigravity 2.0? 🚀"),
    (r"(siapa (kamu|anda)|kamu siapa|nama kamu)",
     "Saya **AntiGravBot**, chatbot yang dibangun untuk menjawab pertanyaan seputar materi "
     "*Google Antigravity 2.0: Era Baru Rekayasa Perangkat Lunak Berbasis Agen AI*."),
    (r"^(bantuan|help|menu|bisa apa|kamu bisa apa)",
     "Saya bisa menjelaskan:\n- Gambaran umum Antigravity 2.0\n- 4 pilar utama "
     "(Desktop Agent Manager, IDE & Ext, CLI & SDK, Agents & Skills)\n- Verifikasi via Artifacts\n"
     "- Editor View vs Manager View\n- 3 tingkat otonomi (Off, Auto, Turbo)\n- Alur kerja, manfaat, dan tips aman"),
]

SYSTEM_PROMPT = (
    "Kamu adalah AntiGravBot, asisten pembelajaran untuk materi kuliah "
    "'Google Antigravity 2.0: Era Baru Rekayasa Perangkat Lunak Berbasis Agen AI'. "
    "Jawab dalam Bahasa Indonesia yang jelas, ramah, dan ringkas (maksimal ±150 kata), "
    "boleh memakai poin atau tabel markdown. Jawab HANYA berdasarkan KONTEKS MATERI yang diberikan. "
    "Jangan mengarang fitur, angka, atau fakta yang tidak ada di konteks. Jika pertanyaan berada di "
    "luar materi, katakan dengan sopan bahwa materi tidak membahasnya, lalu sarankan topik yang tersedia."
)


class ChatbotAntigravity:
    def __init__(self, dokumen, api_key=None, model="gemini-flash-latest", ambang=0.12):
        self.dokumen = list(dokumen)
        self.ambang = ambang
        self.model = model
        self.api_key = api_key
        self._client = None

        korpus = [
            f"{d['judul']} {d['judul']} {d.get('kata_kunci', '')} {d['konten']}"
            for d in self.dokumen
        ]
        self.vec_kata = TfidfVectorizer(preprocessor=normalisasi, ngram_range=(1, 2), sublinear_tf=True)
        self.vec_huruf = TfidfVectorizer(preprocessor=normalisasi, analyzer="char_wb",
                                         ngram_range=(3, 5), sublinear_tf=True)
        self.mat_kata = self.vec_kata.fit_transform(korpus)
        self.mat_huruf = self.vec_huruf.fit_transform(korpus)

    # ------------------------------------------------------------------ LLM
    @property
    def llm_tersedia(self) -> bool:
        return bool(self.api_key)

    def _gemini(self):
        if self._client is None:
            from google import genai  # pip install google-genai
            self._client = genai.Client(api_key=self.api_key)
        return self._client

    # ------------------------------------------------------------ retrieval
    def cari(self, pertanyaan: str, k: int = 3):
        q_kata = self.vec_kata.transform([pertanyaan])
        q_huruf = self.vec_huruf.transform([pertanyaan])
        skor = 0.6 * cosine_similarity(q_kata, self.mat_kata)[0] + \
               0.4 * cosine_similarity(q_huruf, self.mat_huruf)[0]
        urut = np.argsort(skor)[::-1][:k]
        return [(self.dokumen[i], float(skor[i])) for i in urut]

    # --------------------------------------------------------------- jawab
    def jawab(self, pertanyaan: str, riwayat=None, pakai_llm: bool = True) -> dict:
        teks = pertanyaan.strip()
        for pola, balasan in INTENT:
            if re.search(pola, teks.lower()):
                return {"jawaban": balasan, "sumber": [], "mode": "intent", "skor": 1.0}

        hasil = self.cari(teks, k=4)
        top_doc, top_skor = hasil[0]

        if top_skor < self.ambang:
            saran = ", ".join(f"*{d['judul']}*" for d, _ in hasil[:3])
            return {
                "jawaban": "Maaf, sepertinya pertanyaan itu belum dibahas dalam materi Antigravity 2.0. "
                           f"Mungkin maksud Anda salah satu topik ini: {saran}?",
                "sumber": [], "mode": "fallback", "skor": top_skor,
            }

        relevan = [(d, s) for d, s in hasil if s >= max(self.ambang, 0.5 * top_skor)]

        if pakai_llm and self.llm_tersedia:
            try:
                jawaban = self._jawab_llm(teks, relevan, riwayat or [])
                return {"jawaban": jawaban, "sumber": [d["judul"] for d, _ in relevan],
                        "mode": "gemini-rag", "skor": top_skor}
            except Exception as e:  # jaringan/kuota/API key salah -> fallback
                self.error_terakhir = str(e)

        jawaban = f"**{top_doc['judul']}**\n\n{top_doc['konten']}"
        lain = [d["judul"] for d, s in relevan[1:3] if d["kode"] != top_doc["kode"]]
        if lain:
            jawaban += "\n\n🔎 *Topik terkait:* " + " · ".join(lain)
        return {"jawaban": jawaban, "sumber": [top_doc["judul"]], "mode": "retrieval", "skor": top_skor}

    def _jawab_llm(self, pertanyaan, relevan, riwayat):
        from google.genai import types

        konteks = "\n\n".join(f"[{d['kode']}] {d['judul']}\n{d['konten']}" for d, _ in relevan)
        histori = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in riwayat[-6:])
        prompt = (
            f"KONTEKS MATERI:\n{konteks}\n\n"
            f"RIWAYAT PERCAKAPAN:\n{histori or '-'}\n\n"
            f"PERTANYAAN MAHASISWA: {pertanyaan}"
        )
        # Coba model yang diatur, lalu alias "latest" dari Google bila model lama sudah dihentikan.
        kandidat = list(dict.fromkeys([self.model, "gemini-flash-latest", "gemini-flash-lite-latest"]))
        error = None
        for nama in kandidat:
            try:
                resp = self._gemini().models.generate_content(
                    model=nama,
                    contents=prompt,
                    config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0.3),
                )
                if resp.text:
                    self.model = nama          # ingat model yang berhasil
                    return resp.text.strip()
                error = RuntimeError("Respons Gemini kosong")
            except Exception as e:
                error = e
                pesan = str(e).lower()
                # Hanya pindah model jika model tidak ditemukan / tidak diizinkan; error lain langsung dilempar
                if not any(k in pesan for k in ("not found", "404", "not supported", "permission", "403")):
                    raise
        raise error

    # ------------------------------------------------------------ evaluasi
    def evaluasi(self, data_uji):
        """Hitung akurasi top-1 dan top-3 retrieval terhadap data uji."""
        baris, benar1, benar3 = [], 0, 0
        for q, kode in data_uji:
            hasil = self.cari(q, k=3)
            prediksi = [d["kode"] for d, _ in hasil]
            b1, b3 = prediksi[0] == kode, kode in prediksi
            benar1 += b1
            benar3 += b3
            baris.append({"pertanyaan": q, "target": kode, "prediksi": prediksi[0],
                          "skor": round(hasil[0][1], 3), "top1_benar": b1, "top3_benar": b3})
        n = len(data_uji)
        return baris, benar1 / n, benar3 / n

## 4. Konfigurasi (Colab Secrets)

In [ ]:
import os
DATABASE_URL, GEMINI_API_KEY = os.environ.get("DATABASE_URL"), os.environ.get("GEMINI_API_KEY")
GEMINI_MODEL = "gemini-flash-latest"
try:
    from google.colab import userdata
    for k in ("DATABASE_URL", "GEMINI_API_KEY"):
        try:
            globals()[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass
print("Database :", "✅ terisi" if DATABASE_URL else "⚠️ kosong (log tidak disimpan)")
print("Gemini   :", "✅ terisi" if GEMINI_API_KEY else "⚪ kosong (mode Retrieval)")

## 5. Evaluasi akurasi retrieval

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt

bot = ChatbotAntigravity(DOKUMEN, api_key=GEMINI_API_KEY, model=GEMINI_MODEL)
baris, akurasi_top1, akurasi_top3 = bot.evaluasi(DATA_UJI)
df_eval = pd.DataFrame(baris)
print(f"Akurasi Top-1: {akurasi_top1:.0%} | Akurasi Top-3: {akurasi_top3:.0%} | n = {len(df_eval)}")
display(df_eval)

warna = ["#1b6fb3" if b else "#d9534f" for b in df_eval["top1_benar"]]
plt.figure(figsize=(9, 6))
plt.barh(df_eval["pertanyaan"], df_eval["skor"], color=warna)
plt.axvline(bot.ambang, ls="--", color="gray", label=f"ambang = {bot.ambang}")
plt.gca().invert_yaxis(); plt.xlabel("Skor relevansi (cosine)")
plt.title("Skor relevansi per pertanyaan uji (merah = top-1 salah)"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Uji pertanyaan di luar materi -> harus ditolak (mode fallback)
for q in ["resep rendang", "siapa presiden indonesia", "apa itu kubernetes"]:
    r = bot.jawab(q)
    print(f"{q:30s} -> mode={r['mode']:9s} skor={r['skor']:.3f}")

## 6. Demo tanya-jawab

In [ ]:
from IPython.display import Markdown
for q in ["Apa itu Google Antigravity 2.0?", "Apa beda Editor View dan Manager View?",
          "Kapan sebaiknya memakai Mode Turbo?"]:
    r = bot.jawab(q)
    display(Markdown(f"**🧑‍🎓 {q}**\n\n🤖 {r['jawaban']}\n\n`mode: {r['mode']} · relevansi: {r['skor']:.2f}`\n\n---"))

## 7. Koneksi database Supabase / Neon
Membuat tabel, mengisi basis pengetahuan, dan mencoba menyimpan log.

In [ ]:
"""
db.py
Koneksi ke PostgreSQL cloud (Supabase atau Neon) memakai SQLAlchemy.

Tabel:
  kb_antigravity : basis pengetahuan (bisa diedit dari Table Editor Supabase/Neon)
  chat_logs      : riwayat pertanyaan & jawaban pengguna
  feedback       : penilaian 👍/👎 dari pengguna
"""

import json
import re

import pandas as pd
from sqlalchemy import create_engine, text

SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS kb_antigravity (
    id          SERIAL PRIMARY KEY,
    kode        TEXT UNIQUE NOT NULL,
    topik       TEXT NOT NULL,
    judul       TEXT NOT NULL,
    konten      TEXT NOT NULL,
    kata_kunci  TEXT DEFAULT '',
    sumber      TEXT DEFAULT 'Infografis',
    updated_at  TIMESTAMPTZ DEFAULT now()
);

CREATE TABLE IF NOT EXISTS chat_logs (
    id          BIGSERIAL PRIMARY KEY,
    session_id  TEXT NOT NULL,
    pertanyaan  TEXT NOT NULL,
    jawaban     TEXT NOT NULL,
    sumber      TEXT DEFAULT '[]',
    mode        TEXT,
    skor        REAL,
    created_at  TIMESTAMPTZ DEFAULT now()
);

CREATE TABLE IF NOT EXISTS feedback (
    id          BIGSERIAL PRIMARY KEY,
    chat_id     BIGINT REFERENCES chat_logs(id) ON DELETE CASCADE,
    nilai       SMALLINT NOT NULL,          -- 1 = 👍, 0 = 👎
    created_at  TIMESTAMPTZ DEFAULT now()
);

CREATE INDEX IF NOT EXISTS idx_chat_logs_created ON chat_logs (created_at DESC);
"""


def buat_engine(database_url: str):
    url = str(database_url).strip()
    # Ambil hanya bagian postgresql://... (abaikan "psql '...'", tanda kutip, spasi, dll.)
    m = re.search(r"postgres(?:ql)?(?:\+\w+)?://[^\s'\"]+", url)
    if m:
        url = m.group(0)
    if url.startswith("postgres://"):
        url = "postgresql://" + url[len("postgres://"):]
    if url.startswith("postgresql://"):
        url = "postgresql+psycopg2://" + url[len("postgresql://"):]
    connect_args = {}
    if "sslmode" not in url and "localhost" not in url and "127.0.0.1" not in url:
        connect_args["sslmode"] = "require"  # Supabase & Neon wajib SSL
    return create_engine(url, pool_pre_ping=True, pool_size=3, max_overflow=2,
                         pool_recycle=300, connect_args=connect_args)


def init_db(engine):
    with engine.begin() as conn:
        for stmt in SCHEMA_SQL.split(";"):
            if stmt.strip():
                conn.execute(text(stmt))


def seed_kb(engine, dokumen, timpa: bool = False):
    """Isi tabel kb_antigravity. timpa=False -> hanya menambah dokumen yang belum ada,
    sehingga editan manual di Supabase/Neon tidak tertimpa."""
    aksi = """DO UPDATE SET topik=EXCLUDED.topik, judul=EXCLUDED.judul, konten=EXCLUDED.konten,
              kata_kunci=EXCLUDED.kata_kunci, sumber=EXCLUDED.sumber, updated_at=now()""" \
        if timpa else "DO NOTHING"
    sql = text(f"""
        INSERT INTO kb_antigravity (kode, topik, judul, konten, kata_kunci, sumber)
        VALUES (:kode, :topik, :judul, :konten, :kata_kunci, :sumber)
        ON CONFLICT (kode) {aksi}
    """)
    with engine.begin() as conn:
        conn.execute(sql, [{**{"kata_kunci": "", "sumber": "Infografis"}, **d} for d in dokumen])


def load_kb(engine):
    with engine.connect() as conn:
        rows = conn.execute(text(
            "SELECT kode, topik, judul, konten, kata_kunci, sumber FROM kb_antigravity ORDER BY kode"
        )).mappings().all()
    return [dict(r) for r in rows]


def simpan_chat(engine, session_id, pertanyaan, hasil) -> int:
    with engine.begin() as conn:
        return conn.execute(text("""
            INSERT INTO chat_logs (session_id, pertanyaan, jawaban, sumber, mode, skor)
            VALUES (:sid, :q, :a, :src, :mode, :skor) RETURNING id
        """), {"sid": session_id, "q": pertanyaan, "a": hasil["jawaban"],
               "src": json.dumps(hasil["sumber"], ensure_ascii=False),
               "mode": hasil["mode"], "skor": hasil["skor"]}).scalar_one()


def simpan_feedback(engine, chat_id: int, nilai: int):
    with engine.begin() as conn:
        conn.execute(text("INSERT INTO feedback (chat_id, nilai) VALUES (:c, :n)"),
                     {"c": chat_id, "n": nilai})


def ambil_log(engine, limit: int = 500) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text("""
            SELECT c.id, c.created_at, c.session_id, c.pertanyaan, c.jawaban, c.sumber,
                   c.mode, c.skor, f.nilai AS feedback
            FROM chat_logs c
            LEFT JOIN LATERAL (SELECT nilai FROM feedback WHERE chat_id = c.id
                               ORDER BY created_at DESC LIMIT 1) f ON TRUE
            ORDER BY c.created_at DESC LIMIT :n
        """), conn, params={"n": limit})


def sensor(teks: str) -> str:
    """Sembunyikan user/password di pesan error agar tidak tampil di aplikasi publik."""
    teks = re.sub(r"(postgres(?:ql)?(?:\+\w+)?://)[^@\s'\"]+@", r"\1***:***@", str(teks))
    return re.sub(r"password=\S+", "password=***", teks)

In [ ]:
engine = None
if DATABASE_URL:
    engine = buat_engine(DATABASE_URL)
    init_db(engine)
    seed_kb(engine, DOKUMEN, timpa=True)   # timpa=True: samakan isi DB dengan notebook
    kb_db = load_kb(engine)
    print(f"✅ Terhubung. {len(kb_db)} dokumen ada di tabel kb_antigravity.")
    bot = ChatbotAntigravity(kb_db, api_key=GEMINI_API_KEY, model=GEMINI_MODEL)
    display(pd.DataFrame(kb_db)[["kode", "topik", "judul", "sumber"]])
else:
    print("⚠️ DATABASE_URL belum diisi di Colab Secrets — langkah ini dilewati.")

In [ ]:
# Simpan contoh percakapan ke database dan tampilkan log
if engine is not None:
    import uuid
    sid = "colab-" + uuid.uuid4().hex[:6]
    for q in ["Sebutkan 4 pilar utama Antigravity", "Fungsi agents.md dan skills.md?"]:
        cid = simpan_chat(engine, sid, q, bot.jawab(q))
        simpan_feedback(engine, cid, 1)
    display(ambil_log(engine, limit=10)[["created_at", "pertanyaan", "mode", "skor", "feedback"]])

## 8. Chat interaktif
Ketik pertanyaan, ketik `keluar` untuk berhenti.

In [ ]:
import uuid
sid = "colab-" + uuid.uuid4().hex[:6]
riwayat = []
print("🤖 AntiGravBot siap! (ketik 'keluar' untuk berhenti)\n")
while True:
    q = input("🧑‍🎓 Anda: ").strip()
    if q.lower() in ("keluar", "exit", "quit", ""):
        print("🤖 Sampai jumpa!"); break
    r = bot.jawab(q, riwayat=riwayat)
    print(f"\n🤖 AntiGravBot [{r['mode']} · {r['skor']:.2f}]:\n{r['jawaban']}\n")
    riwayat += [{"role": "user", "content": q}, {"role": "assistant", "content": r["jawaban"]}]
    if engine is not None:
        simpan_chat(engine, sid, q, r)

## 9. Publikasi ke Streamlit
Aplikasi web (`app.py`, `engine.py`, `db.py`, `knowledge_base.py`, `requirements.txt`) diunggah ke GitHub
lalu dideploy di **share.streamlit.io** dengan `DATABASE_URL` yang sama di menu *Secrets*.
Karena memakai database yang sama, log dari Colab dan dari aplikasi web tampil bersama di Dashboard.